## Building a Database Agent: From SQL Tool to Natural Language Queries

A database agent lets users ask questions in plain language — "Which book is the most expensive?" — and get back a correct, data-grounded answer. Under the hood, the agent translates the question into SQL, executes it, and formats the result.

Building one that *works reliably* requires solving three problems:

| Problem | Why it's hard | Our solution |
|---|---|---|
| **Safe SQL execution** | The model could generate `DROP TABLE` | Three-layer security: keyword check + comment stripping + read-only connection |
| **Domain knowledge** | The model doesn't know `price` is TEXT, not a number | Per-table SKILL.md files with schema, examples, and gotchas |
| **Scale** | Writing knowledge docs for 50 tables by hand is painful | `generate_skills` script auto-generates drafts |

**What you'll build in this notebook:**

By the end, you will type a question like "哪本书最贵？" and get back a correct answer — powered by a real LLM making real tool calls to your database.

**Before/after — why Skills matter:**

| Question | Without Skills | With Skills |
|---|---|---|
| "哪本书最贵？" | `ORDER BY price DESC` (TEXT sort — wrong) | `ORDER BY CAST(price AS REAL) DESC` (correct) |
| "3月收入多少？" | Includes cancelled orders | Excludes `status = '已取消'` |
| "钻石会员有几个？" | Doesn't know the column name | Reads Skill, filters `member_level` directly |

### 1. Setup

#### 1.1 Install packages

The tutorial sections (2-8) only use the Python standard library. The interactive agent demo in Section 9 uses the `openai` SDK to call your LLM.

In [ ]:
%pip install openai --quiet

#### 1.2 Import modules

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import time
from pathlib import Path
from typing import Any

### 2. Create a sample database

We use a small **bookstore** scenario with 3 tables:

```
customers  <--+
              | customer_id
orders    ----+
              | book_id
books      <--+
```

> **Think of it like this:** `customers` is the address book, `books` is the product catalog, and `orders` is the receipt drawer. The agent needs to open the right drawer for each question.

The database has a few **intentional traps** — the same traps you'll find in real-world databases:

| Trap | What goes wrong |
|---|---|
| `price` is TEXT, not a number | `ORDER BY price DESC` gives `"89" > "168"` (alphabetical!) |
| `status` has "已取消" (cancelled) | Revenue sums include cancelled orders if you forget to filter |
| `member_level` is Chinese text | No numeric rank — the model must know the exact values |

In [ ]:
DB_PATH = Path("sample_bookstore.sqlite")

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# -- customers -------------------------------------------------------
cur.execute(
    "CREATE TABLE customers ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  name        TEXT    NOT NULL,"
    "  email       TEXT    UNIQUE,"
    "  city        TEXT,"
    "  member_level TEXT   DEFAULT '普通',"
    "  created_at  TEXT    DEFAULT (datetime('now'))"
    ")"
)
customers = [
    ("张三", "zhangsan@example.com", "北京", "金卡"),
    ("李四", "lisi@example.com", "上海", "普通"),
    ("王五", "wangwu@example.com", "广州", "银卡"),
    ("赵六", "zhaoliu@example.com", "深圳", "钻石"),
    ("孙七", "sunqi@example.com", "杭州", "普通"),
    ("周八", "zhouba@example.com", "成都", "金卡"),
    ("吴九", "wujiu@example.com", "北京", "银卡"),
    ("郑十", "zhengshi@example.com", "上海", "普通"),
    ("钱十一", "qian11@example.com", "广州", "金卡"),
    ("陈十二", "chen12@example.com", "深圳", "普通"),
]
cur.executemany(
    "INSERT INTO customers (name, email, city, member_level) VALUES (?, ?, ?, ?)",
    customers,
)

# -- books -----------------------------------------------------------
cur.execute(
    "CREATE TABLE books ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  title       TEXT    NOT NULL,"
    "  author      TEXT    NOT NULL,"
    "  genre       TEXT,"
    "  price       TEXT    NOT NULL,"
    "  stock       INTEGER DEFAULT 0,"
    "  publisher   TEXT,"
    "  publish_year INTEGER"
    ")"
)
books = [
    ("三体", "刘慈欣", "科幻", "36.00", 150, "重庆出版社", 2008),
    ("活着", "余华", "文学", "29.00", 200, "作家出版社", 1993),
    ("Python编程从入门到实践", "Eric Matthes", "技术", "89.00", 80, "人民邮电出版社", 2020),
    ("明朝那些事儿", "当年明月", "历史", "168.00", 60, "浙江人民出版社", 2009),
    ("原则", "瑞·达利欧", "经管", "98.00", 45, "中信出版社", 2018),
    ("流浪地球", "刘慈欣", "科幻", "32.00", 120, "中国华侨出版社", 2000),
    ("深度学习", "Ian Goodfellow", "技术", "168.00", 30, "人民邮电出版社", 2017),
    ("百年孤独", "马尔克斯", "文学", "55.00", 90, "南海出版公司", 2011),
    ("人类简史", "尤瓦尔·赫拉利", "历史", "68.00", 75, "中信出版社", 2014),
    ("从零到一", "彼得·蒂尔", "经管", "45.00", 55, "中信出版社", 2015),
    ("设计模式", "GoF", "技术", "79.00", 40, "机械工业出版社", 2000),
    ("围城", "钱钟书", "文学", "25.00", 180, "人民文学出版社", 1947),
]
cur.executemany(
    "INSERT INTO books (title, author, genre, price, stock, publisher, publish_year) "
    "VALUES (?, ?, ?, ?, ?, ?, ?)",
    books,
)

# -- orders ----------------------------------------------------------
cur.execute(
    "CREATE TABLE orders ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  customer_id INTEGER NOT NULL REFERENCES customers(id),"
    "  book_id     INTEGER NOT NULL REFERENCES books(id),"
    "  quantity    INTEGER NOT NULL DEFAULT 1,"
    "  total_price TEXT    NOT NULL,"
    "  order_date  TEXT    NOT NULL,"
    "  status      TEXT    DEFAULT '已完成'"
    ")"
)
orders = [
    (1, 1, 2, "72.00", "2025-01-15", "已完成"),
    (1, 3, 1, "89.00", "2025-02-20", "已完成"),
    (2, 2, 1, "29.00", "2025-01-10", "已完成"),
    (2, 5, 1, "98.00", "2025-03-01", "待发货"),
    (3, 1, 1, "36.00", "2025-02-14", "已完成"),
    (3, 8, 2, "110.00", "2025-03-05", "已完成"),
    (4, 7, 1, "168.00", "2025-01-22", "已完成"),
    (4, 4, 1, "168.00", "2025-02-28", "已取消"),
    (5, 6, 3, "96.00", "2025-03-10", "待发货"),
    (6, 3, 1, "89.00", "2025-01-05", "已完成"),
    (6, 11, 1, "79.00", "2025-02-15", "已完成"),
    (7, 9, 1, "68.00", "2025-03-12", "已完成"),
    (8, 12, 1, "25.00", "2025-01-20", "已完成"),
    (9, 10, 2, "90.00", "2025-02-25", "待发货"),
    (10, 1, 1, "36.00", "2025-03-15", "已完成"),
]
cur.executemany(
    "INSERT INTO orders (customer_id, book_id, quantity, total_price, order_date, status) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    orders,
)

conn.commit()
conn.close()

print(f"Database created: {DB_PATH}")
print(f"  Tables: customers ({len(customers)} rows), books ({len(books)} rows), orders ({len(orders)} rows)")

Quick sanity check — peek at the first few rows of each table:

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

for table in ["customers", "books", "orders"]:
    rows = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchall()
    print(f"\n-- {table} (first 3 rows) --")
    for r in rows:
        print(dict(r))

conn.close()

### 3. Build the `execute_sql` tool

This is the **only way** the agent can talk to the database. Every query goes through this function.

> **Analogy:** Think of `execute_sql` as a bank teller window with bulletproof glass. The customer (LLM) can ask questions and get answers, but can never reach through the window to touch the vault (modify data).

**Three-layer security** — if any one layer fails, the next catches it:

```
Layer 1:  Keyword check      "Does the SQL start with SELECT?"
Layer 2:  Comment stripping   "Is someone hiding DELETE after a -- comment?"
Layer 3:  Read-only mode      "Even if Layers 1-2 fail, SQLite itself refuses writes"
```

**Structured output** — instead of returning raw text, we return a JSON object with `warnings`. When the query returns 0 rows, the warning tells the model "hey, maybe check your WHERE clause" — this is how the agent learns to self-correct.

#### 3.1 Implementation

In [ ]:
MAX_ROWS = 10
MAX_OUTPUT_LENGTH = 50_000
DEFAULT_TIMEOUT = 30

_DANGEROUS_KEYWORDS = (
    "DROP", "TRUNCATE", "DELETE", "ALTER", "CREATE",
    "INSERT", "UPDATE", "REPLACE", "ATTACH", "DETACH",
    "PRAGMA", "VACUUM", "REINDEX", "GRANT", "REVOKE",
)


def _strip_sql_comments(sql: str) -> str:
    """Strip -- and /* */ comments to prevent injection bypass."""
    no_line = re.sub(r"--[^\n]*", "", sql)
    no_block = re.sub(r"/\*.*?\*/", "", no_line, flags=re.DOTALL)
    return no_block


def execute_sql(
    sql: str,
    db_path: str | Path = DB_PATH,
    timeout: int = DEFAULT_TIMEOUT,
    max_rows: int = MAX_ROWS,
) -> dict[str, Any]:
    """
    Execute a read-only SELECT query and return structured results.

    Returns a dict with keys:
      - status: "success" | "error" | "timeout"
      - columns, data, total_rows, row_count, truncated
      - warnings: list of hints for the model
    """
    start = time.time()

    # -- Layer 1: keyword check --
    if not sql or not sql.strip():
        return {"status": "error", "error": "SQL query cannot be empty",
                "duration_ms": int((time.time() - start) * 1000)}

    cleaned_upper = _strip_sql_comments(sql).strip().upper()
    for kw in _DANGEROUS_KEYWORDS:
        if re.match(rf"^{kw}(?:\s|$)", cleaned_upper):
            return {"status": "error",
                    "error": f"Only SELECT queries allowed. Found: {kw}",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}

    if not re.match(r"^(SELECT|WITH)\b", cleaned_upper):
        return {"status": "error",
                "error": "Only SELECT or WITH ... SELECT queries allowed.",
                "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}

    # -- Layer 3: read-only connection --
    connection = None
    try:
        uri = f"file:{db_path}?mode=ro"
        connection = sqlite3.connect(uri, uri=True)
        connection.row_factory = sqlite3.Row

        deadline = time.time() + timeout
        connection.set_progress_handler(
            lambda: 1 if time.time() > deadline else 0, 1000
        )

        cursor = connection.execute(sql)
        col_names = [d[0] for d in cursor.description] if cursor.description else []

        # Use fetchmany instead of fetchall to avoid loading entire
        # result sets into memory on large tables without LIMIT.
        batch = cursor.fetchmany(max_rows + 1)
        truncated = len(batch) > max_rows
        rows_to_convert = batch[:max_rows]
        total_rows = max_rows + 1 if truncated else len(batch)

        data = [dict(r) for r in rows_to_convert]
        duration_ms = int((time.time() - start) * 1000)

        warnings: list[str] = []
        if total_rows == 0:
            warnings.append(
                "Query returned 0 rows. Check: table name, column names, "
                "WHERE conditions, data availability."
            )
        if truncated:
            warnings.append(
                f"Showing {max_rows} rows (more available). "
                f"Add WHERE clauses or LIMIT to narrow results."
            )

        result: dict[str, Any] = {
            "status": "success",
            "sql": sql,
            "columns": col_names,
            "data": data,
            "row_count": len(data),
            "total_rows": total_rows,
            "truncated": truncated,
            "duration_ms": duration_ms,
        }

        # Guard against individual rows with very large TEXT fields
        if len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH:
            while (
                len(data) > 1
                and len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH
            ):
                data = data[:-1]
                result["data"] = data
                result["row_count"] = len(data)
                result["truncated"] = True
            if not any("length limit" in w for w in warnings):
                warnings.append(
                    "Results truncated due to output length limit. "
                    "Select fewer columns or add WHERE clauses."
                )

        if warnings:
            result["warnings"] = warnings
        return result

    except sqlite3.OperationalError as e:
        msg = str(e)
        if "interrupted" in msg.lower():
            return {"status": "timeout",
                    "error": f"Query timed out after {timeout}s",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}
        return {"status": "error", "error": msg, "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    except Exception as e:
        return {"status": "error", "error": str(e), "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    finally:
        if connection:
            connection.close()

#### 3.2 Test the tool

Let's verify it works — both normal queries and security rejection:

In [ ]:
# Normal query
result = execute_sql("SELECT title, price FROM books LIMIT 3")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Empty result triggers a warning hint for the model
result = execute_sql("SELECT * FROM books WHERE genre = '玄幻'")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Security: reject DROP — Layer 1 catches it
result = execute_sql("DROP TABLE books")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Security: reject comment-based bypass — Layer 2 strips the comment, then Layer 1 catches DELETE
result = execute_sql("-- just a comment\nDELETE FROM books")
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 3.3 The TEXT-as-number pitfall

This is the #1 mistake models make. When `price` is stored as TEXT, `ORDER BY price` gives **alphabetical** order — `"89.00" > "168.00"` because `"8" > "1"`.

The fix is simple: `CAST(price AS REAL)`. But the model won't know to do this unless we tell it — that's what Skills are for (Section 4).

In [ ]:
# Wrong: TEXT sort — "89.00" comes before "168.00" lexicographically
result = execute_sql("SELECT title, price FROM books ORDER BY price DESC LIMIT 5")
print("TEXT sort (wrong):")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

print()

# Correct: CAST to REAL for numeric ordering
result = execute_sql("SELECT title, CAST(price AS REAL) AS price FROM books ORDER BY price DESC LIMIT 5")
print("Numeric sort (correct):")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

### 4. Write SKILL.md — teaching the model about your data

Now we have a safe SQL tool, but the model still doesn't know *how* to query correctly. It doesn't know `price` is TEXT, or that cancelled orders should be excluded.

> **Analogy:** Imagine hiring a new data analyst. You wouldn't just give them database access and say "go". You'd hand them a data dictionary, point out the gotchas, and show a few example queries. **That's exactly what a SKILL.md is** — a cheat sheet for the model, one per table.

Skills are [NexAU](https://github.com/nex-agi/NexAU)'s standard mechanism for domain knowledge injection. Each table gets its own SKILL.md file.

#### 4.1 SKILL.md structure

| Section | What it does | Think of it as... |
|---|---|---|
| `description` | Model reads this to decide "do I need this Skill?" | The title on a filing cabinet drawer |
| **When to use** | Example questions that should trigger this Skill | "Open this drawer when someone asks about..." |
| **Schema** | Column names, types, PKs, FKs | The data dictionary |
| **Common values** | Enum values like `status`: `已完成`, `待发货`, `已取消` | A lookup table taped to the desk |
| **Example queries** | 2-3 correct SQL examples | "Here's how the last analyst did it" |
| **Gotchas** | Traps and pitfalls | **The most valuable part** — "Don't sort `price` without CAST!" |

> **Gotchas is the most valuable section.** Models don't fail because they can't find the table — they fail because they don't know about the traps hiding in your data.

#### 4.2 Full example: SKILL.md for the `books` table

In [ ]:
books_skill = '''---
name: books
description: >-
  Use this skill whenever the user asks about books — titles, authors, genres,
  prices, stock levels, or publishers. Join to orders via book_id.
---

# books

The complete book catalog. One row per book, keyed by `id`.

## When to use

- "What science fiction books do we have?"
- "Which book is the most expensive?"
- "How many books by 刘慈欣?"

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Book ID — join key for `orders.book_id` |
| `title` | TEXT | Book title |
| `author` | TEXT | Author name |
| `genre` | TEXT | Category: `文学` / `技术` / `历史` / `科幻` / `经管` |
| `price` | TEXT | Price in yuan — **TEXT not numeric**, use `CAST(price AS REAL)` |
| `stock` | INTEGER | Inventory count |
| `publisher` | TEXT | Publisher name |
| `publish_year` | INTEGER | Year of publication |

## Common values

- `genre`: `文学`, `技术`, `历史`, `科幻`, `经管`

## Example queries

**Most expensive books:**

```sql
SELECT title, author, CAST(price AS REAL) AS price_yuan
FROM books
ORDER BY price_yuan DESC
LIMIT 5;
```

## Gotchas

- `price` is **TEXT**, not REAL — always `CAST(price AS REAL)` for numeric ops.
- `genre` uses Chinese category names. For fuzzy search use `LIKE '%技术%'`.
'''

print(books_skill)

#### 4.3 Anti-patterns

| Don't do this | What goes wrong |
|---|---|
| `description: "Information about books"` | Too vague — model can't decide when to read it |
| Skip Gotchas | Model keeps hitting the TEXT-as-number trap |
| Example queries use `SELECT *` | Model copies the pattern, result sets explode |
| Put all knowledge in system prompt | Wastes tokens on every turn; model's attention gets diluted |
| One Skill for 5 tables | Can't selectively load — all or nothing |

### 5. Auto-generate Skills

Writing SKILL.md by hand is great for 3 tables. For 30 tables, you need automation.

The script below reads any SQLite database and generates a SKILL.md draft for each table. It automatically detects:

- Table structure (columns, types, PK, FK)
- TEXT columns that secretly store numbers (like our `price` field)
- Enum-like columns (few distinct values)
- FK relationships (generates JOIN examples)

Parts that need your human judgment are marked `[TODO]`.

#### 5.1 Helper functions

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    """Return all user table names."""
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    """Return column metadata for a table."""
    # Table names come from sqlite_master, so they are trusted identifiers.
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    """Return FK mapping {local_col: 'ref_table.ref_col'}."""
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    """Return the most common values in a column."""
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    """Detect if a TEXT column actually stores numeric values."""
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False

#### 5.2 Main generator function

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    """Generate a SKILL.md draft for a single table."""
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "(no explicit PK)"

    fk_notes = [f"`{lc}` -> `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK -> `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " | ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            val_str = ", ".join(f"`{v[0]}`" for v in vals)
            common_sections.append(f"- `{col['name']}`: {val_str}")

    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** | PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

#### 5.3 Run on the sample database

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"Found {len(tables)} tables: {', '.join(tables)}\n")

for table in tables:
    print("=" * 60)
    print(f"  {table}")
    print("=" * 60)
    print(generate_skill_md(conn, table))

conn.close()

Notice what the generator caught automatically:

- `books.price` and `orders.total_price` flagged as "TEXT storing numbers" — the #1 gotcha
- FK relationships detected, JOIN examples generated
- Enum columns (`status`, `genre`, `member_level`) had their values listed
- `[TODO]` markers show where you need to add business context

### 6. System Prompt

The system prompt tells the agent *how to work*. Ours defines a **7-step workflow**:

```
Plan -> Read Skills -> Write SQL -> Execute -> Reflect -> Answer
```

> **The key rule:** "ALWAYS read the relevant Skill before writing a query." Without this, the model will skip the Skill and guess column names — and hit every trap in the data.

In [ ]:
SYSTEM_PROMPT = """
You are a database agent. Your job: translate natural-language questions
into correct SQL, execute the query, and return a clear answer grounded in the data.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: `customers`, `books`, `orders`
- Primary join keys: `orders.customer_id` -> `customers.id`, `orders.book_id` -> `books.id`

## Table Knowledge (Skills)

### books
- Columns: id (PK), title, author, genre, price (TEXT!), stock, publisher, publish_year
- genre values: 文学, 技术, 历史, 科幻, 经管
- GOTCHA: `price` is TEXT — use CAST(price AS REAL) for numeric operations

### customers
- Columns: id (PK), name, email, city, member_level, created_at
- member_level values: 普通, 银卡, 金卡, 钻石 (no numeric rank)

### orders
- Columns: id (PK), customer_id (FK->customers), book_id (FK->books), quantity, total_price (TEXT!), order_date, status
- status values: 已完成, 待发货, 已取消
- GOTCHA: `total_price` is TEXT — use CAST(total_price AS REAL)
- GOTCHA: Exclude `status = '已取消'` from revenue calculations
- GOTCHA: `total_price` is pre-computed; don't re-multiply quantity * price

## Workflow

1. **Plan.** Identify which tables you need.
2. **Write the SQL.** SQLite syntax. Always use LIMIT. Prefer explicit columns over SELECT *.
3. **Execute.** Call `execute_sql`.
4. **Reflect.** If 0 rows or warnings, reconsider your query — maybe wrong column name or too-strict filter.
5. **Answer** in the user's language. Be concise. Show the key data.

## Constraints

- Only SELECT queries work — the tool rejects everything else.
- Don't hallucinate columns. If unsure, query the schema first.
"""

print(SYSTEM_PROMPT)

### 7. Run the Agent — natural language queries over your database

Now let's put it all together into a **real, working agent**. No frameworks needed — just the `openai` SDK, our `execute_sql` tool, and a simple loop.

> **How it works:** You type a question in plain language. The LLM reads the system prompt (which contains all the Skills knowledge), decides what SQL to run, calls our `execute_sql` tool, reads the result, and gives you a human-readable answer.

**You need an API key** for any OpenAI-compatible API (OpenAI, DeepSeek, Moonshot, etc.).

#### 7.1 Configure your API

In [ ]:
from openai import OpenAI

# ============================================================
# Fill in your API configuration here
# ============================================================
API_KEY = "sk-xxx"            # Your API key
BASE_URL = "https://api.openai.com/v1"  # Or: https://api.deepseek.com, etc.
MODEL = "gpt-4o-mini"        # Or: deepseek-chat, moonshot-v1-8k, etc.
# ============================================================

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"Client ready: {BASE_URL} / {MODEL}")

#### 7.2 Define the tool schema

This is the "instruction manual" the LLM sees. It tells the model: "you have a tool called `execute_sql` that takes a SQL string and returns results".

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "execute_sql",
            "description": (
                "Execute a read-only SQL query against the bookstore SQLite database. "
                "Only SELECT queries are allowed. Results are capped at max_rows. "
                "Always use LIMIT. Use CAST for TEXT columns that store numbers."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "The SQL query to execute (SELECT only)."
                    },
                    "max_rows": {
                        "type": "integer",
                        "description": "Maximum rows to return. Default 10.",
                        "default": 10
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

#### 7.3 The agent loop

This is the core — a simple loop that:
1. Sends the conversation to the LLM
2. If the LLM wants to call a tool, execute it and feed the result back
3. Repeat until the LLM gives a final text answer

> **This is exactly what NexAU does internally**, but here we do it in ~30 lines so you can see every step.

In [ ]:
def ask(question: str, verbose: bool = True) -> str:
    """
    Ask the database agent a question in natural language.
    Returns the agent's final answer.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    if verbose:
        print(f"Q: {question}")
        print("-" * 50)

    for turn in range(10):  # max 10 tool-call rounds
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            temperature=0.1,
        )

        msg = response.choices[0].message

        # If no tool calls, the model is giving its final answer
        if not msg.tool_calls:
            answer = msg.content or "(no response)"
            if verbose:
                print(f"\nA: {answer}")
            return answer

        # Process each tool call
        messages.append(msg)  # add assistant message with tool_calls

        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"  [Tool Call] {fn_name}({json.dumps(fn_args, ensure_ascii=False)})")

            if fn_name == "execute_sql":
                result = execute_sql(**fn_args)
            else:
                result = {"status": "error", "error": f"Unknown tool: {fn_name}"}

            if verbose:
                status = result.get("status", "?")
                rows = result.get("row_count", "?")
                print(f"  [Result]    status={status}, rows={rows}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "(max turns reached)"

#### 7.4 Try it out!

Let's ask some questions. Watch the tool calls — you'll see the model reading the data and self-correcting:

In [ ]:
# Simple query — does the model use CAST for price?
ask("哪本书最贵？")

In [ ]:
# Aggregation with filter — does it exclude cancelled orders?
ask("2025年3月的总收入是多少？")

In [ ]:
# Cross-table JOIN
ask("哪个客户消费最多？列出前3名")

In [ ]:
# Test domain knowledge — does it know member_level values?
ask("钻石会员有几个？分别是谁？")

#### 7.5 Ask your own questions!

Try modifying the question below. Some ideas:
- "哪个城市的客户最多？"
- "刘慈欣的书总共卖了多少钱？"
- "库存最少的3本书是什么？"
- "每个月的订单量分别是多少？" 

In [ ]:
# Change this to any question you want!
ask("每种类型的书平均价格是多少？")

### 8. Summary

You built a complete database agent from scratch:

| Step | What you built | Why it matters |
|---|---|---|
| `execute_sql` | Three-layer secure SQL executor | Model can query but never modify data |
| SKILL.md | Per-table domain knowledge | Model avoids TEXT-as-number traps, knows to filter cancelled orders |
| `generate_skills` | Auto-generates Skill drafts | Scales from 3 tables to 300 |
| System Prompt | 5-step workflow | Model plans, executes, reflects, then answers |
| Agent Loop | LLM + tool calling | Natural language in, correct answers out |

**To use this with your own database:**

1. Replace `DB_PATH` with your SQLite file
2. Run `generate_skills` to auto-generate SKILL.md drafts
3. Fill in the `[TODO]` markers with business context
4. Update the `SYSTEM_PROMPT` with your table knowledge
5. Ask questions!

**To scale to production with [NexAU](https://github.com/nex-agi/NexAU):**

The agent loop above is ~30 lines. NexAU adds session persistence, streaming, multi-agent orchestration, middleware, and tracing. But the core pattern is the same — the code you wrote here is the foundation.

In [ ]:
# Cleanup
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Cleaned up {DB_PATH}")